In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def summarize_run(run_dir, tail_n=5):
    run_dir = Path(run_dir)

    with open(run_dir / "args.json", encoding="utf-8") as f:
        args = json.load(f)

    fid_path = run_dir / "validation_fid.jsonl"
    fids = []

    if fid_path.exists():
        with open(fid_path, encoding="utf-8") as f:
            fids = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

    row = {
        "run": run_dir.name,
        **{f"arg_{key}": value for key, value in args.items()},
        "fid_best": np.nan,
        "fid_best_iteration": np.nan,
        "fid_final": np.nan,
        "fid_final_iteration": np.nan,
        "fid_tail_median": np.nan,
        "fid_tail_mean": np.nan,
        "fid_tail_max": np.nan,
        "fid_tail_n": 0,
    }

    if fids:
        values = np.array([item["fid"] for item in fids], dtype=float)
        iterations = np.array([item["iteration"] for item in fids])

        tail = values[-tail_n:]

        best_index = np.argmin(values)

        row.update({
            "fid_best": values[best_index],
            "fid_best_iteration": iterations[best_index],
            "fid_final": values[-1],
            "fid_final_iteration": iterations[-1],
            "fid_tail_median": np.median(tail),
            "fid_tail_mean": np.mean(tail),
            "fid_tail_max": np.max(tail),
            "fid_tail_n": len(tail),
        })

    return row


# One experiment directory containing timestamped run directories
experiment_root = Path("experiments_dcgan")

df = pd.DataFrame(
    summarize_run(run_dir, tail_n=5)
    for run_dir in experiment_root.iterdir()
    if run_dir.is_dir() and (run_dir / "args.json").exists()
)

df = df.sort_values("fid_tail_median")
print(df)

For comparing DiffAugment policies:

comparison = (
    df.groupby("arg_diff_aug_policy", dropna=False)
    .agg(
        runs=("run", "count"),
        median_tail_fid=("fid_tail_median", "median"),
        median_best_fid=("fid_best", "median"),
        median_final_fid=("fid_final", "median"),
        worst_tail_fid=("fid_tail_max", "max"),
    )
    .sort_values("median_tail_fid")
)

print(comparison)